# Working

**Request:** [T04] Clean these orders up and total them

> New email from Sai Suram <sai@agents.agentstore.it.com>
> Subject: [T04] Clean these orders up and total them
> Thread ID: AAQkADI1N2Y5MTE3LTE1MDctNGY0Yy1iYzQ5LWEzNmE5NzAyYzk4NQAQAGhnD0Ss7mtDm2NhT3wSS8M=

Each cell below is one run in the sandbox, in the order it happened, with whatever it printed and produced.


In [1]:
import pandas as pd
import io

data = """order_id,customer,order_date,amount,qty
1001,Acme Corp,2026-01-15,"1,250.00",5
1002,acme corp ,15/01/2026,"980.50",3
1003,Beta Ltd,2026-02-03,"2,100.00",8
1003,Beta Ltd,2026-02-03,"2,100.00",8
1004,BETA LTD,03/02/2026,"450.25",2
1005,Gamma Inc,2026-02-20,"1,875.00",-4
1006,Gamma Inc,20/02/2026,"730.00",1"""

df = pd.read_csv(io.StringIO(data))

# Store original rows for comparison later
original_rows = len(df)

# 1. Standardize customer names
df['customer'] = df['customer'].str.title().str.strip()

# 2. Convert 'order_date' to datetime, handling mixed formats
# Attempt YYYY-MM-DD first, then DD/MM/YYYY
df['order_date'] = pd.to_datetime(df['order_date'], errors='coerce', format='%Y-%m-%d')
df['order_date'] = df['order_date'].fillna(pd.to_datetime(df['order_date'], errors='coerce', format='%d/%m/%Y'))

# 3. Clean 'amount' column
df['amount'] = df['amount'].replace('[$,]', '', regex=True).astype(float)

# 4. Handle negative 'qty' (assuming it should be positive, correcting invalid entries)
# For this analysis, negative quantity will make the revenue negative, which is acceptable if it represents a return.
# However, if it's an error, we might want to flag or correct. The prompt asks to 'clean it up', 
# so we'll assume it's an error for now and set to 0 or absolute value, let's go with absolute to keep a record of the transaction.
# Or, if revenue is amount * qty, negative qty means negative revenue, which is a valid business scenario for returns.
# Let's keep it as is for now and ensure revenue calculation is correct.
# If the negative quantity is an error in data entry, it would need further clarification.
# For now, I will assume it represents a return and will process it as-is for revenue calculation.
# If 'cleaned' implies fixing errors, a negative quantity could be an error. Let's make it absolute value for 'cleaned rows' if it is an error.
# The request implies 'clean up' so I'll make the quantities positive as an assumed correction for data entry errors.
# The only explicitly mentioned fix is 'mess', and '-4' is messy. I will change it to 4.
fixes = []
if (df['qty'] < 0).any():
    fixes.append("Corrected negative quantity in order 1005 from -4 to 4, assuming data entry error.")
    df['qty'] = df['qty'].abs()


# 5. Handle duplicates - The prompt mentions '1003,Beta Ltd,2026-02-03,"2,100.00",8' appears twice.
# We need to decide if this is a true duplicate (same order entry twice) or two items on the same order.
# Given 'order_id' is repeated, and all other fields are identical, this is likely a duplicate entry.
# I will remove duplicates based on all columns.
initial_rows = len(df)
df.drop_duplicates(inplace=True)
duplicate_rows_removed = initial_rows - len(df)
if duplicate_rows_removed > 0:
    fixes.append(f"Removed {duplicate_rows_removed} duplicate row(s).")

# Calculate total revenue for each row (amount * qty)
df['revenue'] = df['amount'] * df['qty']

# Calculate revenue by customer
revenue_by_customer = df.groupby('customer')['revenue'].sum().reset_index()
revenue_by_customer.columns = ['Customer', 'Total Revenue']

# Prepare output for Excel
cleaned_df_output = df[['order_id', 'customer', 'order_date', 'amount', 'qty', 'revenue']]

# Convert order_date back to string for consistent Excel output if needed, or keep as datetime
# For Excel, datetime objects are fine, but ensure format is readable.
cleaned_df_output['order_date'] = cleaned_df_output['order_date'].dt.strftime('%Y-%m-%d')

# Create an Excel writer object
output_file_path = "/tmp/output/cleaned_orders_and_revenue.xlsx"
with pd.ExcelWriter(output_file_path, engine='xlsxwriter') as writer:
    cleaned_df_output.to_excel(writer, sheet_name='Cleaned Orders', index=False)
    revenue_by_customer.to_excel(writer, sheet_name='Revenue by Customer', index=False)


# Output fixes to stdout
print("Fixes Made:")
for fix in fixes:
    print(f"- {fix}")

print("\nRevenue by Customer:")
print(revenue_by_customer.to_string(index=False))


Fixes Made:
- Corrected negative quantity in order 1005 from -4 to 4, assuming data entry error.
- Removed 1 duplicate row(s).

Revenue by Customer:
 Customer  Total Revenue
Acme Corp         9191.5
 Beta Ltd        17700.5
Gamma Inc         8230.0



[files written: cleaned_orders_and_revenue.xlsx]
